## Data Cleaning

Some columns will be dropped, and some will be transformed

### Columns that will be largely unchanged (dropping rows with missing values):
 - Ranked
 - Popularity
 - Members
 - Favorites
 - Watching
 - Completed
 - On-hold
 - Dropped
 - Plan to watch
 - Episodes

### Columns to be transformed:
 - Genres (One hot encoding)
 - Type (One hot encoding)
 - Source (One hot encoding)
 - Rating (One hot encoding)
 - Aired (extracting the year)
 - Duration (extracting the seconds)

### Columns that will be dropped:
 - MAL_ID (id variable is not helpful in prediction)
 - English name (there is already a name variable)
 - Japanese name (there is already a name variable)
 - Premiered (air date is already present)
 - Name (not helpful for models)
 - Producers (string variable)
 - Licensors (string variable)
 - Studios (string variable)
 - Ranked (too many missing values)
 - Score columns (these would just enable score prediction)
   - Score-10
   - Score-9
   - Score-8
   - Score-7
   - Score-6
   - Score-5
   - Score-4
   - Score-3
   - Score-2
   - Score-1


Additionally, some rows will be filled in with averages to ensure there are no missing values

### Import Data and Packages

In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [25]:
df = pd.read_csv("anime.csv")
df = df[df['Score'] != "Unknown"]

In [26]:
df.columns

Index(['MAL_ID', 'Name', 'Score', 'Genres', 'English name', 'Japanese name',
       'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors',
       'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity',
       'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped',
       'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6',
       'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1'],
      dtype='object')

In [27]:
col_drop = cols = ["MAL_ID","English name","Japanese name","Premiered","Name","Producers","Licensors","Studios", "Ranked", "Score-10","Score-9","Score-8","Score-7","Score-6","Score-5","Score-4","Score-3","Score-2","Score-1"]
df.drop(col_drop, axis = 1, inplace = True)

### One hot encoding Columns

In [28]:
cols = ["Type", "Source", "Rating"]
for col in cols:
    dummies = pd.get_dummies(df[col], prefix=col)
    dummies = dummies.astype(int)
    df = df.join(dummies)
    df.drop(columns=[col], inplace=True)


In [29]:
#cleaning the genres column
df['Genres'] = df['Genres'].astype(str).str.strip().str.split(r'\s*,\s*')
one_hot = df['Genres'].explode().str.get_dummies().groupby(level=0).sum()
df.drop("Genres", axis=1, inplace = True)
df = df.join(one_hot)

### Air Date Cleaning (Extracting the air year)

In [30]:
#Dealing with the aired column now
df["Aired"].unique()

array(['Apr 3, 1998 to Apr 24, 1999', 'Sep 1, 2001',
       'Apr 1, 1998 to Sep 30, 1998', ..., 'Feb 11, 2021', 'Feb 12, 2021',
       'Mar 14, 2021'], shape=(9207,), dtype=object)

In [31]:
df['Year'] = df['Aired'].str.extract(r'(\d{4})').astype('float').astype('Int64')

In [32]:
df.drop("Aired", axis = 1, inplace = True)

### Cleaning the Duration column (length in seconds)

In [33]:
#dealing with the duration column
df.Duration.head()

0    24 min. per ep.
1      1 hr. 55 min.
2    24 min. per ep.
3    25 min. per ep.
4    23 min. per ep.
Name: Duration, dtype: object

In [34]:
import re

def extract_seconds(s):
    s = str(s).lower()

    hr_match  = re.search(r'(\d+)\s*hr', s)
    min_match = re.search(r'(\d+)\s*min', s)
    sec_match = re.search(r'(\d+)\s*sec', s)

    hours = int(hr_match.group(1)) if hr_match else 0
    minutes = int(min_match.group(1)) if min_match else 0
    seconds = int(sec_match.group(1)) if sec_match else 0

    total_seconds = hours*3600 + minutes*60 + seconds

    return total_seconds if total_seconds > 0 else None

In [35]:
df["Seconds"] = df["Duration"].apply(extract_seconds)

In [36]:
df.drop("Duration", axis =1, inplace = True)
print("Dropped Duration")

Dropped Duration


### Dealing with missing values or wrong data types

In [37]:
df.columns

Index(['Score', 'Episodes', 'Popularity', 'Members', 'Favorites', 'Watching',
       'Completed', 'On-Hold', 'Dropped', 'Plan to Watch', 'Type_Movie',
       'Type_Music', 'Type_ONA', 'Type_OVA', 'Type_Special', 'Type_TV',
       'Source_4-koma manga', 'Source_Book', 'Source_Card game',
       'Source_Digital manga', 'Source_Game', 'Source_Light novel',
       'Source_Manga', 'Source_Music', 'Source_Novel', 'Source_Original',
       'Source_Other', 'Source_Picture book', 'Source_Radio', 'Source_Unknown',
       'Source_Visual novel', 'Source_Web manga', 'Rating_G - All Ages',
       'Rating_PG - Children', 'Rating_PG-13 - Teens 13 or older',
       'Rating_R - 17+ (violence & profanity)', 'Rating_R+ - Mild Nudity',
       'Rating_Rx - Hentai', 'Rating_Unknown', 'Action', 'Adventure', 'Cars',
       'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game',
       'Harem', 'Hentai', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic',
       'Martial Arts', 'Mecha', 'Military', '

In [38]:
for col in ['Popularity', 'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped', 'Plan to Watch', "Episodes"]:
    df = df[df[col] != "Unknown"]
    df[col] = df[col].astype(float)

In [39]:
df = df.dropna(subset = ["Year", "Seconds"])


## Train Test Splitting

In [40]:
score = df["Score"]
X = df.drop("Score", axis=1)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, score, test_size=0.2, random_state=42)

In [42]:
X_train.to_csv("X_train_data.csv", index = False)
y_train.to_csv("Y_train_data.csv", index = False)
X_test.to_csv("X_test_data.csv", index = False)
y_test.to_csv("Y_test_data.csv", index = False)